In [8]:
import networkx as nx
import polars as pl
from pyvis.network import Network

In [2]:
# 1) Load the dataset from the specific year in Parquet format: 
path = "../data/processed/trade_2024.parquet"
df = pl.read_parquet(path)
df

Year,Exporter,Importer,Product_Code,Value_Thousands_USD
u16,cat,cat,u32,f32
2024,"""AFG""","""AGO""",80810,0.176
2024,"""AFG""","""AGO""",330499,2.295
2024,"""AFG""","""AGO""",732510,5.617
2024,"""AFG""","""AGO""",848330,2.42
2024,"""AFG""","""AGO""",853610,0.605
…,…,…,…,…
2024,"""ZMB""","""URY""",610429,0.585
2024,"""ZMB""","""URY""",848490,0.048
2024,"""ZMB""","""URY""",870810,0.02


In [3]:
# 2) Group by country pairs to add up the total value of trade and prevent Networkx from overlapping each edge: 
df = df.group_by(["Exporter", "Importer"]).agg(
    pl.col("Value_Thousands_USD").sum().alias("weight")
)
df

Exporter,Importer,weight
cat,cat,f32
"""AZE""","""AGO""",314.787994
"""AZE""","""CIV""",0.942
"""ARG""","""BHS""",2871.431152
"""IDN""","""BOL""",49592.578125
"""KWT""","""CHN""",1.1721551e7
…,…,…
"""TTO""","""POL""",42401.402344
"""TTO""","""BFA""",0.141
"""ARE""","""MMR""",47511.261719


In [ ]:
# 3) Create the graph as a directed graph (Exporter -> Importer) and the edges weighted by its trade value:
graph = nx.from_pandas_edgelist(
    df.to_pandas(),
    source="Exporter",
    target="Importer",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

In [13]:
# 4) Use the PyVis library in order to visualize the graph: 
net = Network(notebook=True, directed=True, bgcolor="#222222", font_color="white")
net.from_nx(graph)
net.show("world_trade_network.html")

world_trade_network.html


In [14]:
# 5) Use the Gephi software tto visualize in local the graph: 
nx.write_gexf(graph, "world_trade_network.gexf")